In [79]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ======================
# SETUP
# ======================
os.makedirs("models", exist_ok=True)

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)


# ======================
# PREPROCESSING
# ======================
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Price'] = df['Price'].ffill()

# Log transformation
df['Price_Log'] = np.log(df['Price'])

# ======================
# FEATURE ENGINEERING
# ======================

# Lags
df['Lag1'] = df['Price_Log'].shift(1)
df['Lag2'] = df['Price_Log'].shift(2)
df['Lag3'] = df['Price_Log'].shift(3)
df['Lag5'] = df['Price_Log'].shift(5)
df['Lag10'] = df['Price_Log'].shift(10)

# Moving averages
df['MA7'] = df['Price_Log'].rolling(7).mean()
df['MA14'] = df['Price_Log'].rolling(14).mean()
df['MA30'] = df['Price_Log'].rolling(30).mean()

# Volatility
df['Volatility7'] = df['Price_Log'].rolling(7).std()
df['Volatility14'] = df['Price_Log'].rolling(14).std()

# Momentum
df['Momentum'] = df['Price_Log'] - df['Price_Log'].shift(5)

# Target (log return)
df['Log_Return'] = df['Price_Log'] - df['Lag1']

# Drop NA
df = df.dropna()

# ======================
# SPLIT
# ======================
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

features = [
    'Lag1', 'Lag2', 'Lag3', 'Lag5', 'Lag10',
    'MA7', 'MA14', 'MA30',
    'Volatility7', 'Volatility14',
    'Momentum'
]

X_train = train[features]
y_train = train['Log_Return']

X_test = test[features]
y_test = test['Price']   # REAL price

# ======================
# MODEL
# ======================
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# ======================
# PREDICTION (CORRECT)
# ======================
pred_log_return = rf.predict(X_test)

# Convert back to price (NON-RECURSIVE, stable)
pred_log_price = X_test['Lag1'].values + pred_log_return
pred_price = np.exp(pred_log_price)

# ======================
# EVALUATION
# ======================
print("=== Random Forest ===")
print(f"MAE  : {mean_absolute_error(y_test, pred_price):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred_price)):.4f}")
print(f"R2   : {r2_score(y_test, pred_price):.4f}")
print(f"MAPE : {np.mean(np.abs((y_test - pred_price) / y_test)) * 100:.4f}")

# ======================
# SAVE MODEL + FEATURES
# ======================
joblib.dump(rf, "models/random_forest_log.pkl")

# SAVE FEATURE LIST (IMPORTANT FIX)
joblib.dump(features, "models/features.pkl")

print("Model and features saved successfully.")

=== Random Forest ===
MAE  : 602.4360
RMSE : 946.3370
R2   : 0.9979
MAPE : 0.6926
Model and features saved successfully.


In [45]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ======================
# SETUP
# ======================
os.makedirs("models", exist_ok=True)

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ======================
# CLEAN DATA
# ======================
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Price'] = df['Price'].ffill()

# ======================
# LOG TRANSFORM
# ======================
df['Price_Log'] = np.log(df['Price'])

# ======================
# FEATURE ENGINEERING
# ======================

# Lags (ALL based on LOG PRICE)
df['Lag1'] = df['Price_Log'].shift(1)
df['Lag2'] = df['Price_Log'].shift(2)
df['Lag3'] = df['Price_Log'].shift(3)
df['Lag5'] = df['Price_Log'].shift(5)
df['Lag10'] = df['Price_Log'].shift(10)

# Moving averages
df['MA7'] = df['Price_Log'].rolling(7).mean()
df['MA14'] = df['Price_Log'].rolling(14).mean()
df['MA30'] = df['Price_Log'].rolling(30).mean()

# Volatility
df['Volatility7'] = df['Price_Log'].rolling(7).std()
df['Volatility14'] = df['Price_Log'].rolling(14).std()

# Momentum
df['Momentum'] = df['Price_Log'] - df['Price_Log'].shift(5)

# Target
df['Log_Return'] = df['Price_Log'].diff().rolling(3).mean()

# Drop NaN AFTER everything
df = df.dropna()

# ======================
# SPLIT
# ======================
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

features = [
    'Lag1', 'Lag2', 'Lag3', 'Lag5', 'Lag10',
    'MA7', 'MA14', 'MA30',
    'Volatility7', 'Volatility14',
    'Momentum'
]

X_train = train[features]
y_train = train['Log_Return']

X_test = test[features]
y_test = test['Price']   # REAL PRICE for evaluation

# ======================
# MODEL
# ======================
gb = GradientBoostingRegressor(
    n_estimators=50,      # ↓ fewer trees
    learning_rate=0.1,    # ↑ too aggressive
    max_depth=2,          # ↓ less complex
    random_state=42
)

gb.fit(X_train, y_train)

# ======================
# PREDICTION (CORRECT FIX)
# ======================
pred_log_return = gb.predict(X_test)

# reconstruct log price
pred_log_price = X_test['Lag1'].values + pred_log_return

# convert to real price
pred_price = np.exp(pred_log_price)

# ======================
# EVALUATION
# ======================
print("=== Gradient Boosting (Fixed Log Model) ===")
print(f"MAE  : {mean_absolute_error(y_test, pred_price):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred_price)):.4f}")
print(f"R2   : {r2_score(y_test, pred_price):.4f}")
print(f"MAPE : {np.mean(np.abs((y_test - pred_price) / y_test)) * 100:.4f}")

# ======================
# SAVE MODEL
# ======================
joblib.dump(gb, "models/gradient_boosting_log.pkl")

=== Gradient Boosting (Fixed Log Model) ===
MAE  : 493.5397
RMSE : 784.1628
R2   : 0.9985
MAPE : 0.5665


['models/gradient_boosting_log.pkl']

In [46]:
import pandas as pd
import numpy as np
import joblib
import os
import random
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ======================
# SEED
# ======================
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Price'] = df['Price'].ffill()

# ======================
# LOG TRANSFORM (IMPORTANT)
# ======================
df['Price_Log'] = np.log(df['Price'])

# ======================
# LOG RETURN (MATCH RF + GB)
# ======================
df['Log_Return'] = df['Price_Log'] - df['Price_Log'].shift(1)
df = df.dropna()

# ======================
# SCALE LOG RETURN (IMPORTANT FOR LSTM)
# ======================
scaler = MinMaxScaler()
scaled_log_return = scaler.fit_transform(df[['Log_Return']])

joblib.dump(scaler, "models/lstm_log_scaler.pkl")

# ======================
# WINDOWING
# ======================
def create_window(data, window=60):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_window(scaled_log_return, window=60)

X = X.reshape(X.shape[0], X.shape[1], 1)

# ======================
# SPLIT
# ======================
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ======================
# MODEL
# ======================
model = Sequential([
    Input(shape=(60, 1)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

model.fit(X_train, y_train, epochs=30, batch_size=32, verbose=1)

# ======================
# PREDICTION
# ======================
pred_scaled = model.predict(X_test)

pred_log_return = scaler.inverse_transform(pred_scaled).flatten()
actual_log_return = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# ======================
# RECONSTRUCTION TO PRICE
# ======================
last_log_price = df['Price_Log'].iloc[-len(pred_log_return)-1:-1].values

pred_log_price = last_log_price + pred_log_return
pred_price = np.exp(pred_log_price)

actual_price = df['Price'].iloc[-len(pred_price):].values

# ======================
# EVALUATION
# ======================
print("=== LSTM===")
print(f"MAE  : {mean_absolute_error(actual_price, pred_price):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(actual_price, pred_price)):.4f}")
print(f"R2   : {r2_score(actual_price, pred_price):.4f}")
print(f"MAPE : {np.mean(np.abs((actual_price - pred_price) / actual_price)) * 100:.4f}")

# ======================
# SAVE MODEL
# ======================
model.save("models/lstm_log_model.keras")

Epoch 1/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.0226
Epoch 2/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0055
Epoch 3/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0054
Epoch 4/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0055
Epoch 5/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0055
Epoch 6/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0055
Epoch 7/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0055
Epoch 8/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0055
Epoch 9/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0055
Epoch 10/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0055
Epoch 11/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.0055
Epoch 12/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.0055
Epoch 13/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0055
Epoch 14/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0055
Epoch 15/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0055
Epoc

In [74]:
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df['Price'] = df['Price'].ffill()
df['Price_Log'] = np.log(df['Price'])
df['Log_Return'] = df['Price_Log'].diff()

df['Lag1'] = df['Price_Log'].shift(1)
df['Lag2'] = df['Price_Log'].shift(2)
df['Lag3'] = df['Price_Log'].shift(3)
df['Lag5'] = df['Price_Log'].shift(5)
df['Lag10'] = df['Price_Log'].shift(10)

df['MA7'] = df['Price_Log'].rolling(7).mean()
df['MA14'] = df['Price_Log'].rolling(14).mean()
df['MA30'] = df['Price_Log'].rolling(30).mean()

df['Volatility7'] = df['Price_Log'].rolling(7).std()
df['Volatility14'] = df['Price_Log'].rolling(14).std()

df['Momentum'] = df['Price_Log'] - df['Price_Log'].shift(5)

# ======================
# ALIGN TEST SET
# ======================
split = int(len(df) * 0.8)
df_train = df.iloc[:split]
df_test = df.iloc[split:].reset_index(drop=True)

test_log_price = df_test['Price_Log'].values
test_log_return = df_test['Log_Return'].values

# ======================
# RF PREDICTION
# ======================

rf_pred = rf.predict(df_test[features])

# ======================
# LSTM PREDICTION (already aligned carefully)
# ======================
lstm_pred = model.predict(X_test).flatten()
lstm_pred = scaler.inverse_transform(lstm_pred.reshape(-1,1)).flatten()

# ======================
# ALIGN LENGTH
# ======================
min_len = min(len(rf_pred), len(lstm_pred), len(test_log_return))

rf_pred = rf_pred[:min_len]
lstm_pred = lstm_pred[:min_len]
y_true = test_log_return[:min_len]

# ======================
# META FEATURES
# ======================
hybrid_X = np.column_stack([
    rf_pred,
    lstm_pred,
    rf_pred - lstm_pred,
    (rf_pred + lstm_pred) / 2,
    rf_pred * lstm_pred
])

# ======================
# TRAIN META MODEL (USE TIME SPLIT ONLY)
# ======================
from sklearn.ensemble import GradientBoostingRegressor

meta_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
meta_model.fit(hybrid_X, y_true)
hybrid_pred = meta_model.predict(hybrid_X)

# Remove: y_test_final = y_true[split:] is unused and incorrectly sliced

# ======================
# RECONSTRUCT PRICE (CORRECT WAY)
# ======================
last_log_price = df_test['Price_Log'].values

pred_log_price = df_test['Lag1'].values[:len(hybrid_pred)] + hybrid_pred
actual_log_price = df_test['Lag1'].values[:len(y_true)] + y_true

pred_price = np.exp(pred_log_price)
actual_price = np.exp(actual_log_price)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(actual_price, pred_price)
rmse = np.sqrt(mean_squared_error(actual_price, pred_price))
r2 = r2_score(actual_price, pred_price)
mape = np.mean(np.abs((actual_price - pred_price) / actual_price)) * 100

print("=== CLEAN HYBRID (FIXED ALIGNMENT) ===")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.4f}")

# ======================
# SAVE MODEL
# ======================
joblib.dump(meta_model, "models/hybrid_log.pkl")

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
=== CLEAN HYBRID (FIXED ALIGNMENT) ===
MAE  : 343.5250
RMSE : 488.9693
R2   : 0.9994
MAPE : 0.4070


['models/hybrid_log.pkl']